# Lesson 22 Lab — Packaging an INT4 Inference Deliverable

**Puzzle:** What files make a quantized model reproducible rather than merely loadable on one machine?

This notebook keeps the RTX 5090 outputs from a complete run. Read the theory cells, make a prediction, and then use **Run All** on your own GPU.


## Why this matters

A quantized model is deployable only when its bytes and interpretation travel together. Packed weights without scales are meaningless; correct weights with the wrong tokenizer or base revision are unsafe; and an artifact without a checksum cannot be distinguished from a partial copy. Packaging is therefore part of inference correctness, not administrative cleanup.


## 0. Predict before running

1. List the minimum fields needed to load, validate, and roll back a quantized artifact.
2. Predict the packed payload size for the notebook's weight and scale tensors.
3. Explain what a SHA-256 digest proves and what semantic errors it cannot detect.

For each answer, name the observation that would prove you wrong.


## 1. Name the concrete objects

A deployable package binds tensor shards, scales/zero points, shapes and packing schema, base/tokenizer revisions, runtime requirements, checksums, smoke vectors, and rollback identity.

- A deliverable binds base revision, quantization recipe, tokenizer, tensor shapes, scales, packing, and runtime requirements.
- Checksums detect corruption but do not validate semantics.
- A smoke test and rollback pointer belong beside the artifact.


## 2. Derive the mechanism

A cryptographic hash verifies bytes, while a schema verifies meaning. Both are needed: identical shapes with the wrong scale axis can be semantically corrupt yet perfectly hash-consistent.

A package contract binds schema version, base revision, quantization format, group size/axis, tensor shapes, scale dtype, runtime/backend requirement, checksums, and rollback target. The checksum establishes byte identity; the schema establishes how those bytes should be decoded. Both are needed.

Production packages also include tokenizer/config files, special-token policy, architecture code revision, licenses, model card, and quality/performance reports. Keeping a minimal manifest in the lab makes the invariant testable without publishing checkpoint data.


## 3. Verify the execution environment

The next cell asserts CUDA availability, fixes the seed, locates the lesson, and prints a sanitized GPU/PyTorch/CUDA record. Check it before interpreting output.


In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "22-int4-inference-package"
device = require_cuda()
torch.manual_seed(2026 + 22)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 4. Freeze the comparison

| Role | This run |
|---|---|
| Baseline | unversioned in-memory reference weights with no handoff contract |
| Candidate | temporary packed INT4 payload plus validated manifest and checksum |
| Held constant | fixed tensor shape, group size, serializer, required-field set, rollback ID |
| Measurements | payload bytes, SHA-256, required-field completeness, cleanup status |
| Evidence | `pytorch-gpu` |

**Experiment:** Create an in-memory synthetic INT4 shard on CUDA, serialize only a tiny temporary payload, verify its checksum and manifest fields, then delete the temporary file.


## 5. Read the experiment code

The lab creates a tiny temporary packed payload, hashes and validates its manifest, and deletes it so no model checkpoint enters the repository.

The notebook quantizes a 256×512 matrix, serializes codes and scales into a temporary file, computes SHA-256, records size and interpretation fields, validates required keys, and lets the temporary payload disappear after the check. Only the small manifest evidence remains public.

The exercise proves packaging logic without committing weights. It does not claim compatibility with SafeTensors, Hugging Face quantization configs, or a named production runtime.

Only after these variables match the protocol should the cell be executed.


In [2]:
import hashlib, tempfile
w=torch.randn(256,512,device=device); q,scales,_=symmetric_quantize(w,bits=4,group_size=64); payload=q.cpu().numpy().tobytes()+scales.cpu().numpy().tobytes()
with tempfile.NamedTemporaryFile() as f:
    f.write(payload); f.flush(); digest=hashlib.sha256(Path(f.name).read_bytes()).hexdigest(); size=Path(f.name).stat().st_size
manifest={"schema":1,"format":"reference-int4","group_size":64,"shape":list(w.shape),"sha256":digest,"bytes":size,
          "base_revision":"example-frozen-revision","runtime":"pytorch-reference","rollback":"bf16-baseline-v1"}
required={"schema","format","group_size","shape","sha256","bytes","base_revision","runtime","rollback"}
result=base_result(22,"pytorch-gpu"); result.update({"manifest":manifest,"manifest_complete":required.issubset(manifest),
    "temporary_payload_deleted_after_check":True,"conclusion":"A small packaging contract and checksum were validated without publishing checkpoint data."})


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| Manifest complete | yes |
| Payload bytes | 139,264 bytes |
| Format | reference-int4 |
| Group size | 64 |
| SHA-256 | `bd46d808…6714de` |
| Temporary payload removed | yes |


## 7. Interpret rather than merely print

The generated reference payload was 139,264 bytes and received digest `bd46d8…714de`. Every required manifest field was present, including base revision, format, group size, shape, runtime, and BF16 rollback target. The temporary payload was deleted after validation.

This is a reproducibility and safety result: another process can verify identity and interpretation metadata. It is not a model export, engine load, or distribution license decision.

**Inspection rule:** The lab validates packaging logic; it does not publish a model checkpoint.


## 8. Keep the evidence label honest

This run is labeled **`pytorch-gpu`**. The measured tensors and operations ran on CUDA through PyTorch. The result does not name a separate production backend unless an operator trace identifies it.

The next cell writes the complete structured result; its existing saved output is part of the checked-in evidence.


In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "conclusion": "A small packaging contract and checksum were validated without publishing checkpoint data.",
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gib": 31.358,
    "python": "3.12.13",
    "torch": "2.12.0"
  },
  "evidence_label": "pytorch-gpu",
  "executed_at_utc": "2026-08-07T14:46:06+00:00",
  "lesson": 22,
  "manifest": {
    "base_revision": "example-frozen-revision",
    "bytes": 139264,
    "format": "reference-int4",
    "group_size": 64,
    "rollback": "bf16-baseline-v1",
    "runtime": "pytorch-reference",
    "schema": 1,
    "sha256": "bd46d8080937e2d0becaf647a75b0b1d03d2f54599c0203ed5357a6e126714de",
    "shape": [
      256,
      512
    ]
  },
  "manifest_complete": true,
  "schema_version": 1,
  "temporary_payload_deleted_after_check": true
}
Saved: artifacts/rtx5090-result.json


## 9. Make the bounded decision

> Ship a versioned contract with hashes, schema, compatibility, smoke test, and rollback—not a loose weight file.

**Acceptance/rollback:** Test fresh-environment load, hash verification, schema validation, deterministic smoke output, memory budget, native operator, and rollback artifact before release.

**Failure analysis:** A digest cannot detect that the wrong scale axis was declared if both producer and consumer share the same bad schema. Mutable model tags and missing tokenizer revisions also break reproducibility. Never place secrets, local paths, proprietary weights, or unlicensed datasets in a public package to make a tutorial appear complete.


## 10. Extend the evidence

Define a JSON Schema for the manifest, add per-file hashes and total-size checks, and write a loader that rejects an incompatible runtime or base revision before allocating GPU memory. Test truncation, swapped scale files, wrong group size, and rollback loading as deliberate failure cases.

The full derivation, reproduction command, evidence boundary and primary references are in [`README.md`](README.md).
